# Latihan Quiz Comvis
Nama : Fadhlan Nur Rachman
NIM : 2802491690

In [1]:
# import Library & Dataset

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

TRAIN_PATH = "Dataset/train/"
TEST_PATH = "Dataset/test/"
MODEL_PATH = "orl_lbph_model.xml"

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

In [2]:
# preprocessing face
def preprocess_face(face_img):
    face_img = cv2.resize(face_img,(100, 100))
    face_img = cv2.equalizeHist(face_img)
    return face_img

In [3]:
# detect and crop face
def detect_crop_face(img_gray):
    detected_faces = face_cascade.detectMultiScale(img_gray, scaleFactor=1.05, minNeighbors=2, minSize=(30,30))

    if len(detected_faces) < 1:
        # ORL dataset images are already cropped faces — fallback to full image
        face_img = preprocess_face(img_gray)
        h, w = img_gray.shape
        return face_img, (0, 0, w, h)
    
    x,y,w,h = max(detected_faces, key=lambda rect:rect[2]*rect[3])
    face_img = img_gray[y:y+h, x:x+w]
    face_img = preprocess_face(face_img)
    face_box = (x, y, w, h)

    return face_img, face_box

In [4]:
def load_data(folder_path, class_names=None):
    face_list = []
    label_list = []

    if class_names is None:
        class_names = sorted(os.listdir(folder_path))

    for label, person_name in enumerate(class_names):
        person_folder = os.path.join(folder_path, person_name)

        if not os.path.isdir(person_folder):
            print(f"Warning {person_folder} is not a directory, skipping!")
            continue

        for img_name in sorted(os.listdir(person_folder)):
            if not img_name.lower().endswith((".pgm", ".jpg", ".jpeg", ".png", ".bmp")):
                print(f"Warning {img_name} is not supported on image format, skipping")
                continue

            img_path = os.path.join(person_folder, img_name)
            img_gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img_gray is None:
                print(f"could not read {img_path}")
                continue
            
            face_img, _ = detect_crop_face(img_gray)

            if face_img is None:
                print(f"Warning, Face not detected in {img_path}")
                continue

            face_list.append(face_img)
            label_list.append(label)
    
    return face_list, np.array(label_list, dtype=np.int32), class_names

In [5]:
def train_test_model():
    face_recognizer = cv2.face.LBPHFaceRecognizer_create()
    print("Loading Training Data...")
    train_faces, train_labels, class_names = load_data(TRAIN_PATH)

    if len(train_faces) == 0:
        print("No Data training found!")
        return
    
    print("Training Models...")
    face_recognizer.train(train_faces, train_labels)
    face_recognizer.save(MODEL_PATH)
    print("Model saved to: ", MODEL_PATH)

    print("Loading Test Data...")
    test_faces, test_labels, _ = load_data(TEST_PATH, class_names)

    if len(test_faces) == 0:
        print("No Data test found!")
        return
    
    correct_predictions = 0
    total_predictions = len(test_faces)

    for face_img, true_label in zip(test_faces, test_labels):
        predicted_label, confidence = face_recognizer.predict(face_img)
        
        true_name = class_names[true_label]
        predicted_name = class_names[predicted_label]

        if predicted_label == true_label:
            correct_predictions += 1
            status = "RIGHT"
        else:
            status = "WRONG"
        
        print("Actual       :", true_name)
        print("Predicted    :", predicted_name)
        print(f"Confidence   : {confidence:.2f} | {status}")
    
    accuracy = (correct_predictions / total_predictions) * 100
    print(f"Average Accuracy: {accuracy:.2f}%")

In [6]:
def load_saved_model():
    if not os.path.exists(MODEL_PATH):
        print("MODEL NOT FOUND!, Please train model first")

    face_recognizer = cv2.face.LBPHFaceRecognizer_create()
    face_recognizer.read(MODEL_PATH)

    class_names = sorted(os.listdir(TRAIN_PATH))
    return face_recognizer, class_names

In [7]:
def predict_picture():
    path_input = input("Enter the path to the image for testing: ")
    
    face_recognizer, class_names = load_saved_model()
    if face_recognizer is None:
        return
    
    img_color = cv2.imread(path_input)
    img_gray = cv2.imread(path_input, cv2.IMREAD_GRAYSCALE)

    if img_gray is None:
        print("Image not found or cant be read")
        return
    
    face_img, face_box = detect_crop_face(img_gray)

    if face_img is None:
        print("No face detected in image")
        return
    
    predicted_label, confidence = face_recognizer.predict(face_img)

    x,y,w,h = face_box
    threshold = 70

    if confidence < threshold:
        predicted_name = class_names[predicted_label]
        color = (0, 255, 0)
    else:
        predicted_name = "Unknown"
        color = (0, 0, 255)

    print(f"{path_input} detected as {predicted_name}")
    print(f"Detected subject: {predicted_name}")
    print(f"Detected face location: x={x}, y={y}, w={w}, h={h}")
    print(f"Confidence: {confidence:.2f}")
    print("Note: lower LBPH confidence means better match.")

    display_img = img_color.copy()

    cv2.rectangle(display_img, (x,y), (x+w, y+h), color, 2)
    text = f"{predicted_name} | {confidence}"
    cv2.putText(display_img, text, (x, max(y - 10,20)), cv2.FONT_HERSHEY_COMPLEX, 0.6, color, 2)

    cv2.imshow("Prediciton Result", display_img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    

In [ ]:
def menu():
    while True:
        print("\n=== Face Recognition Menu ===")
        print("\n1. Train and Test Models")
        print("2. Predict")
        print("3. Exit")

        choice = input("Enter your choice: ")
        if choice == '1':
            train_test_model()
        elif choice == '2':
            predict_picture()
        elif choice == '3':
            print("Exit...")
            return
        else:
            print("WRONG INPUT PLEASE ENTER YOUR CHOICE")


menu()


=== Face Recognition Menu ===

1. Train and Test Models
2. Predict
3. Exit
Loading Training Data...
Training Models...
Model saved to:  orl_lbph_model.xml
Loading Test Data...
Actual       : s1
Predicted    : s1
Confidence   : 81.17 | RIGHT
Actual       : s1
Predicted    : s1
Confidence   : 78.61 | RIGHT
Actual       : s1
Predicted    : s1
Confidence   : 75.47 | RIGHT
Actual       : s10
Predicted    : s10
Confidence   : 70.16 | RIGHT
Actual       : s10
Predicted    : s10
Confidence   : 52.17 | RIGHT
Actual       : s10
Predicted    : s40
Confidence   : 88.93 | WRONG
Actual       : s11
Predicted    : s11
Confidence   : 57.28 | RIGHT
Actual       : s11
Predicted    : s11
Confidence   : 53.22 | RIGHT
Actual       : s11
Predicted    : s11
Confidence   : 58.97 | RIGHT
Actual       : s12
Predicted    : s12
Confidence   : 65.31 | RIGHT
Actual       : s12
Predicted    : s12
Confidence   : 57.13 | RIGHT
Actual       : s12
Predicted    : s12
Confidence   : 53.86 | RIGHT
Actual       : s13
Predic